# Notebook 6: Entanglement & Bell Inequality Violation

## What are we exploring?

This notebook tackles a central question in quantum foundations: **can quantum
correlations be explained by local hidden variables satisfying the CHSH assumptions?**

We will:
1. Construct a two-mode squeezed vacuum as the number-basis output of PDC
2. Construct all four polarization Bell states
3. Verify their entanglement using concurrence and von Neumann entropy
4. Implement the CHSH inequality test
5. Show that $S = 2\sqrt{2} > 2$, **violating** the classical bound

## Tensor products and correlations

Bell tests use tensor-product structure, local measurement operators, and
correlation functions. The physical result is that entangled photon correlations
cannot be explained by local hidden-variable/local-realist models satisfying the
CHSH assumptions, while still respecting no-signaling.

## Conventions used in this notebook

- Polarization qubits use $|H\rangle=|0\rangle$ and $|V\rangle=|1\rangle$.
- Analyzer angles are physical polarization angles, so observables use $\cos(2\phi)$ and $\sin(2\phi)$.
- The two-mode squeeze convention is $S_2(\xi)=\exp(\xi^*a_1a_2-\xi a_1^\dagger a_2^\dagger)$.
- QuTiP's `squeezing(a1, a2, z)` helper has a `1/2` in the exponent; match with `z=2*xi`.


## Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import qutip

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16,
    'figure.figsize': (8, 5), 'figure.dpi': 150, 'savefig.dpi': 300,
    'text.usetex': False, 'mathtext.fontset': 'stix', 'font.family': 'STIXGeneral',
})

# Pauli matrices
sx = qutip.sigmax()
sy = qutip.sigmay()
sz = qutip.sigmaz()


def sigma_pol(phi):
    """Polarization analyzer at physical angle phi.
    The corresponding Bloch-sphere angle is 2*phi.
    """
    return np.cos(2*phi) * sz + np.sin(2*phi) * sx


print("Setup complete.")

Setup complete.


## Two-Mode Squeezed Vacuum

Type-I parametric down-conversion at higher gain produces a two-mode squeezed
vacuum:

$$|\mathrm{TMSV}\rangle = \hat{S}_2(\xi)|0,0\rangle
= \frac{1}{\cosh r}\sum_{n=0}^{\infty}(-e^{i\phi_{\rm squeeze}}\tanh r)^n |n,n\rangle.$$

The key signature is perfect photon-number correlation: the two modes contain
the same photon number in every occupied Fock component.